# Modeling Exam (Elite) — CoderPad Practice Problems

If you can solve these cold in 30 min, you're overqualified. Multiple interacting systems, careful algorithmic thinking, tricky edge cases. All formulas provided.

Only attempt after MEDIUM and HARD feel comfortable.

**Rules:** 30 min per problem. No peeking at references. If tests fail, debug — don't skip.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math
from typing import Optional, Tuple, Dict, List
from collections import defaultdict, OrderedDict

---

## Problem 1: Complete Diffusion Training Pipeline

### Scenario

Build the full DDPM pipeline end-to-end: noise schedule, UNet, forward noising, training step, and reverse sampling. Everything must work together.

### Requirements

**`make_schedule(T, beta_start, beta_end) → dict`**
- Linear beta schedule: `betas = linspace(beta_start, beta_end, T)`
- `alphas = 1 - betas`
- `alpha_bar[t] = prod(alphas[0:t+1])` (cumulative product)
- Also store: `sqrt_alpha_bar`, `sqrt_one_minus_alpha_bar`, `sqrt_recip_alpha = 1/sqrt(alphas)`, `posterior_variance = betas * (1 - alpha_bar_prev) / (1 - alpha_bar)` where `alpha_bar_prev[0] = 1.0`
- All values should be float32 tensors of shape `(T,)`

**`SimpleUNet(nn.Module)`**
- Input: `(x, t_emb)` where `x` is `(B, 1, H, W)` and `t_emb` is `(B, time_dim)`
- Constructor: `__init__(self, time_dim=32)`
- Architecture:
  - `time_mlp`: Linear(time_dim, 64) → ReLU (projects time embedding for injection)
  - Down path: `down1` = Conv2d(1, 32, 3, padding=1) → ReLU, then MaxPool2d(2)
  - Down path: `down2` = Conv2d(32, 64, 3, padding=1) → ReLU, then MaxPool2d(2)
  - Middle: `mid` = Conv2d(64, 64, 3, padding=1) → ReLU. **Add** time projection (reshaped to B,64,1,1) to middle output.
  - Up path: Upsample(scale_factor=2) → `up1` = Conv2d(64+64, 32, 3, padding=1) → ReLU (concat skip from down2)
  - Up path: Upsample(scale_factor=2) → `up2` = Conv2d(32+32, 32, 3, padding=1) → ReLU (concat skip from down1)
  - Output: `out` = Conv2d(32, 1, 1) (1×1 conv, no activation)
- Returns predicted noise with same shape as input `x`

**`q_sample(x0, t, schedule, noise=None) → x_t`**
- Forward diffusion: `x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * noise`
- `t` is a batch of integer timesteps `(B,)`. Index into schedule tensors.
- If `noise` is None, sample from N(0, I)
- Reshape schedule values for broadcasting: `(B,)` → `(B, 1, 1, 1)` for images

**`training_step(model, x0, schedule, optimizer) → loss`**
- Sample random `t ~ Uniform{0, ..., T-1}` for each item in batch
- Sample noise `~ N(0, I)`
- Compute `x_t = q_sample(x0, t, schedule, noise)`
- Create time embedding: use sinusoidal encoding. `t_emb = sinusoidal_embedding(t, dim=32)`
  - `sinusoidal_embedding(t, dim)`: half = dim//2, `freqs = exp(-ln(10000) * arange(half) / half)`, `args = t[:, None] * freqs[None, :]`, return `cat([sin(args), cos(args)], dim=-1)` → `(B, dim)`
- Predict: `noise_pred = model(x_t, t_emb)`
- Loss: `MSE(noise_pred, noise)`
- Backprop + optimizer step (zero_grad, backward, step)
- Return loss value (float)

**`sample(model, schedule, shape, T, device='cpu') → x`**
- Start from `x = randn(shape)`
- For `t` from `T-1` down to `0`:
  - Create batch of timestep `t` values and compute `t_emb`
  - `noise_pred = model(x, t_emb)`
  - `x = sqrt_recip_alpha[t] * (x - betas[t] / sqrt_one_minus_alpha_bar[t] * noise_pred)`
  - If `t > 0`: `x = x + sqrt(posterior_variance[t]) * randn_like(x)`
- Return `x`

In [ ]:
def sinusoidal_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    """Sinusoidal time embedding. t: (B,) integer timesteps → (B, dim)."""
    # YOUR CODE HERE
    pass


def make_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02) -> dict:
    """Precompute all DDPM noise schedule tensors."""
    # YOUR CODE HERE
    pass


class SimpleUNet(nn.Module):
    def __init__(self, time_dim: int = 32):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass


def q_sample(x0: torch.Tensor, t: torch.Tensor, schedule: dict, noise: Optional[torch.Tensor] = None) -> torch.Tensor:
    """Forward diffusion process."""
    # YOUR CODE HERE
    pass


def training_step(model: nn.Module, x0: torch.Tensor, schedule: dict, optimizer: torch.optim.Optimizer) -> float:
    """One training step: noise, predict, loss, backprop."""
    # YOUR CODE HERE
    pass


def sample(model: nn.Module, schedule: dict, shape: tuple, T: int, device: str = "cpu") -> torch.Tensor:
    """DDPM reverse sampling from pure noise."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 1 ---
torch.manual_seed(42)

# === Sinusoidal embedding ===
t_test = torch.tensor([0, 5, 49])
emb = sinusoidal_embedding(t_test, dim=32)
assert emb.shape == (3, 32), f"Embedding shape wrong: {emb.shape}"
assert not torch.isnan(emb).any(), "Embedding has NaNs"
# Different timesteps should produce different embeddings
assert not torch.allclose(emb[0], emb[1]), "Different timesteps gave same embedding"

# === Schedule ===
T = 50
sched = make_schedule(T, beta_start=1e-4, beta_end=0.02)
assert sched["betas"].shape == (T,), f"Betas shape: {sched['betas'].shape}"
assert sched["alpha_bar"].shape == (T,), f"Alpha_bar shape: {sched['alpha_bar'].shape}"
# Betas are increasing (linear schedule)
assert (sched["betas"][1:] >= sched["betas"][:-1]).all(), "Betas should be non-decreasing"
# Alpha_bar is decreasing
assert (sched["alpha_bar"][1:] <= sched["alpha_bar"][:-1]).all(), "Alpha_bar should be non-increasing"
# Alpha_bar starts near 1 and ends near 0
assert sched["alpha_bar"][0] > 0.99, f"Alpha_bar[0] should be near 1: {sched['alpha_bar'][0]}"
assert sched["alpha_bar"][-1] < sched["alpha_bar"][0], "Alpha_bar should decrease"
# sqrt versions are consistent
assert torch.allclose(sched["sqrt_alpha_bar"] ** 2, sched["alpha_bar"], atol=1e-6), "sqrt_alpha_bar^2 != alpha_bar"
assert torch.allclose(sched["sqrt_one_minus_alpha_bar"] ** 2, 1 - sched["alpha_bar"], atol=1e-6)
# Posterior variance shape
assert sched["posterior_variance"].shape == (T,), "Posterior variance wrong shape"

# === SimpleUNet forward pass ===
torch.manual_seed(42)
model = SimpleUNet(time_dim=32)
B, H, W = 4, 16, 16
x_in = torch.randn(B, 1, H, W)
t_emb = sinusoidal_embedding(torch.randint(0, T, (B,)), dim=32)
out = model(x_in, t_emb)
assert out.shape == (B, 1, H, W), f"UNet output shape wrong: {out.shape}"
# Gradient flow
loss = out.sum()
loss.backward()
has_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in model.parameters())
assert has_grad, "No gradients flowing through UNet"

# === q_sample ===
torch.manual_seed(42)
model.zero_grad()
x0 = torch.randn(B, 1, H, W)
t_zero = torch.zeros(B, dtype=torch.long)
x_noised_t0 = q_sample(x0, t_zero, sched)
# At t=0, alpha_bar ≈ 1, so x_t ≈ x0
assert torch.allclose(x_noised_t0, x0, atol=0.05), "q_sample at t=0 should be close to x0"
# At large t, x_t should be mostly noise
t_large = torch.full((B,), T - 1, dtype=torch.long)
x_noised_large = q_sample(x0, t_large, sched)
# Variance of noised image should be closer to 1 (noise) than original
assert x_noised_large.std() > 0.5, "Heavily noised image should have std > 0.5"

# === Training step ===
torch.manual_seed(42)
model2 = SimpleUNet(time_dim=32)
opt = torch.optim.Adam(model2.parameters(), lr=1e-3)
x_train = torch.randn(8, 1, 16, 16)
losses = []
for _ in range(5):
    l = training_step(model2, x_train, sched, opt)
    losses.append(l)
assert isinstance(losses[0], float), "training_step should return a float"
assert losses[-1] < losses[0], f"Loss should decrease: {losses[0]:.4f} → {losses[-1]:.4f}"

# === Sampling ===
torch.manual_seed(42)
model2.eval()
with torch.no_grad():
    samples = sample(model2, sched, (2, 1, 16, 16), T)
assert samples.shape == (2, 1, 16, 16), f"Sample shape wrong: {samples.shape}"
assert not torch.isnan(samples).any(), "Samples contain NaNs"
assert not torch.isinf(samples).any(), "Samples contain Infs"

print("Problem 1: ALL TESTS PASSED")

---

## Problem 2: Vision Transformer (ViT) from Scratch

### Scenario

Build a complete Vision Transformer. This requires patch embedding with positional encoding, multi-head self-attention, a transformer encoder stack, and a classification head — all from scratch using only `nn.Linear`, `nn.LayerNorm`, and `nn.Parameter`.

### Requirements

**`PatchEmbedding(nn.Module)`**
- Constructor: `__init__(self, img_size=32, patch_size=8, in_channels=3, embed_dim=128)`
- `num_patches = (img_size // patch_size) ** 2`
- Use `nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)` to extract patches (equivalent to flatten + linear but more efficient)
- `cls_token`: learnable `nn.Parameter` of shape `(1, 1, embed_dim)`
- `pos_embed`: learnable `nn.Parameter` of shape `(1, num_patches + 1, embed_dim)` — +1 for CLS
- Forward: `(B, C, H, W)` → conv → `(B, embed_dim, H', W')` → flatten spatial → transpose to `(B, num_patches, embed_dim)` → prepend CLS token (expand to batch) → add positional embedding → output `(B, num_patches + 1, embed_dim)`

**`TransformerBlock(nn.Module)`**
- Constructor: `__init__(self, embed_dim=128, num_heads=4, mlp_ratio=4.0, dropout=0.0)`
- Pre-norm architecture: `x = x + Attention(LayerNorm(x))` then `x = x + MLP(LayerNorm(x))`
- Attention: `nn.Linear(embed_dim, 3 * embed_dim)` for Q, K, V projection → split into heads → scaled dot-product attention → concat → `nn.Linear(embed_dim, embed_dim)` output projection
  - `scale = (embed_dim // num_heads) ** -0.5`
  - `attn = softmax(Q @ K^T * scale) @ V`
- MLP: `Linear(embed_dim, hidden) → GELU → Dropout → Linear(hidden, embed_dim) → Dropout` where `hidden = int(embed_dim * mlp_ratio)`

**`TransformerEncoder(nn.Module)`**
- Constructor: `__init__(self, embed_dim=128, depth=4, num_heads=4, mlp_ratio=4.0, dropout=0.0)`
- Stack of `depth` TransformerBlocks
- Forward: apply each block sequentially

**`ViT(nn.Module)`**
- Constructor: `__init__(self, img_size=32, patch_size=8, in_channels=3, num_classes=10, embed_dim=128, depth=4, num_heads=4)`
- Pipeline: PatchEmbedding → TransformerEncoder → LayerNorm → extract CLS token (`[:, 0]`) → Linear(embed_dim, num_classes)
- Forward: `(B, 3, 32, 32)` → `(B, num_classes)`

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size: int = 32, patch_size: int = 8, in_channels: int = 3, embed_dim: int = 128):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim: int = 128, num_heads: int = 4, mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass


class TransformerEncoder(nn.Module):
    def __init__(self, embed_dim: int = 128, depth: int = 4, num_heads: int = 4, mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass


class ViT(nn.Module):
    def __init__(self, img_size: int = 32, patch_size: int = 8, in_channels: int = 3,
                 num_classes: int = 10, embed_dim: int = 128, depth: int = 4, num_heads: int = 4):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests for Problem 2 ---
torch.manual_seed(42)

B = 4
img_size, patch_size, in_channels = 32, 8, 3
embed_dim, num_classes = 128, 10
num_patches = (img_size // patch_size) ** 2  # 16

# === PatchEmbedding ===
patch_emb = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
x_img = torch.randn(B, in_channels, img_size, img_size)
patches = patch_emb(x_img)
assert patches.shape == (B, num_patches + 1, embed_dim), f"PatchEmbedding output shape: {patches.shape}, expected {(B, num_patches + 1, embed_dim)}"
# CLS token should be the same across batch (before adding pos embed, but after it's expanded)
# Position embeddings should make each position unique
assert patch_emb.pos_embed.shape == (1, num_patches + 1, embed_dim), f"pos_embed shape: {patch_emb.pos_embed.shape}"
assert patch_emb.cls_token.shape == (1, 1, embed_dim), f"cls_token shape: {patch_emb.cls_token.shape}"

# === TransformerBlock ===
block = TransformerBlock(embed_dim, num_heads=4)
seq_in = torch.randn(B, num_patches + 1, embed_dim)
seq_out = block(seq_in)
assert seq_out.shape == seq_in.shape, f"TransformerBlock changed shape: {seq_out.shape}"
# Residual connection: output should not be zero even with random init
assert seq_out.abs().sum() > 0, "TransformerBlock output is all zeros"

# === Full ViT ===
torch.manual_seed(42)
vit = ViT(img_size=32, patch_size=8, in_channels=3, num_classes=num_classes, embed_dim=embed_dim, depth=4, num_heads=4)
logits = vit(x_img)
assert logits.shape == (B, num_classes), f"ViT output shape: {logits.shape}"
assert logits.dtype == torch.float32, f"ViT output dtype: {logits.dtype}"

# Gradient flow through entire model
loss = logits.sum()
loss.backward()
# Check gradients flow to patch embedding conv
patch_conv_has_grad = False
for name, p in vit.named_parameters():
    if p.grad is not None and p.grad.abs().sum() > 0:
        patch_conv_has_grad = True
        break
assert patch_conv_has_grad, "No gradients in any ViT parameters"

# CLS token aggregates: changing a patch should affect output
torch.manual_seed(42)
vit.zero_grad()
x1 = torch.randn(1, 3, 32, 32)
x2 = x1.clone()
x2[:, :, :8, :8] = torch.randn(1, 3, 8, 8)  # modify one patch
with torch.no_grad():
    out1 = vit(x1)
    out2 = vit(x2)
assert not torch.allclose(out1, out2, atol=1e-5), "Modifying a patch should change ViT output (CLS aggregation)"

# Parameter count sanity check (should have a reasonable number of params)
total_params = sum(p.numel() for p in vit.parameters())
assert total_params > 10000, f"ViT has too few parameters: {total_params}"
assert total_params < 10_000_000, f"ViT has too many parameters: {total_params}"

print("Problem 2: ALL TESTS PASSED")

---

## Problem 3: Flow Matching

### Scenario

Implement the flow matching framework — the newer alternative to DDPM. Instead of predicting noise at discrete timesteps, flow matching learns a continuous velocity field that transports noise to data along straight paths.

### Requirements

**`compute_conditional_flow(x0, x1, t) → x_t`**
- Linear interpolation between data `x0` and noise `x1`
- Formula: `x_t = (1 - t) * x0 + t * x1`
- `t` is `(B, 1, 1, 1)` for broadcasting with `(B, C, H, W)` images
- At `t=0`, returns `x0`. At `t=1`, returns `x1`.

**`compute_velocity_target(x0, x1) → v`**
- The target velocity (tangent to the straight path)
- Formula: `v = x1 - x0`
- Shape: same as `x0`

**`FlowMatchingVelocityNet(nn.Module)`**
- A simple CNN that predicts velocity given `(x_t, t)`
- Constructor: `__init__(self, channels=1, time_dim=32)`
- Architecture: sinusoidal time embedding → time MLP (Linear → ReLU → Linear(time_dim, 64))
- Conv path: Conv2d(channels, 64, 3, padding=1) → ReLU → Conv2d(64, 64, 3, padding=1) → ReLU → Conv2d(64, channels, 3, padding=1)
- Inject time: add time_mlp output reshaped to (B, 64, 1, 1) after the first conv+ReLU
- Input: `(x_t, t)` where `x_t` is `(B, C, H, W)` and `t` is `(B,)` float in `[0, 1]`
- Output: predicted velocity, same shape as `x_t`

**`flow_matching_loss(model, x0, x1=None) → loss`**
- If `x1` is None, sample `x1 ~ N(0, I)` (noise)
- Sample `t ~ Uniform(0, 1)` as `(B,)` floats
- Compute `x_t = compute_conditional_flow(x0, x1, t.view(B, 1, 1, 1))`
- Compute `v_target = compute_velocity_target(x0, x1)`
- Predict `v_pred = model(x_t, t)`
- Return `MSE(v_pred, v_target)`

**`ode_sample(model, shape, num_steps, device='cpu') → x`**
- Start from noise: `x = randn(shape)` at `t = 1.0`
- Euler integration from `t=1` to `t=0` in `num_steps` steps
- `dt = 1.0 / num_steps`
- For each step: `t_current` goes from `1 - dt` down to `0` (or equivalently, integrate backward)
- Actually, since velocity points from x0 to x1 (data to noise), we integrate backward:
  - `x = x - dt * model(x, t_tensor)` where `t_tensor` starts at `1.0` and decreases by `dt`
- Return final `x` (generated data)

In [ ]:
def compute_conditional_flow(x0: torch.Tensor, x1: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """Linear interpolation: x_t = (1-t)*x0 + t*x1."""
    # YOUR CODE HERE
    pass


def compute_velocity_target(x0: torch.Tensor, x1: torch.Tensor) -> torch.Tensor:
    """Target velocity: v = x1 - x0."""
    # YOUR CODE HERE
    pass


class FlowMatchingVelocityNet(nn.Module):
    def __init__(self, channels: int = 1, time_dim: int = 32):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass


def flow_matching_loss(model: nn.Module, x0: torch.Tensor, x1: Optional[torch.Tensor] = None) -> torch.Tensor:
    """Compute flow matching training loss."""
    # YOUR CODE HERE
    pass


def ode_sample(model: nn.Module, shape: tuple, num_steps: int, device: str = "cpu") -> torch.Tensor:
    """Generate samples via Euler ODE integration from t=1 (noise) to t=0 (data)."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 3 ---
torch.manual_seed(42)

B, C, H, W = 4, 1, 16, 16
x0 = torch.randn(B, C, H, W)
x1 = torch.randn(B, C, H, W)

# === Flow interpolation boundary conditions ===
t_zero = torch.zeros(B, 1, 1, 1)
t_one = torch.ones(B, 1, 1, 1)
xt_at_0 = compute_conditional_flow(x0, x1, t_zero)
xt_at_1 = compute_conditional_flow(x0, x1, t_one)
assert torch.allclose(xt_at_0, x0, atol=1e-6), "At t=0, x_t should equal x0"
assert torch.allclose(xt_at_1, x1, atol=1e-6), "At t=1, x_t should equal x1"

# Midpoint should be average
t_half = torch.full((B, 1, 1, 1), 0.5)
xt_mid = compute_conditional_flow(x0, x1, t_half)
assert torch.allclose(xt_mid, (x0 + x1) / 2, atol=1e-6), "At t=0.5, x_t should be midpoint"

# === Velocity target ===
v = compute_velocity_target(x0, x1)
assert v.shape == x0.shape, f"Velocity shape wrong: {v.shape}"
assert torch.allclose(v, x1 - x0, atol=1e-6), "Velocity should be x1 - x0"

# === VelocityNet forward ===
torch.manual_seed(42)
fm_model = FlowMatchingVelocityNet(channels=C, time_dim=32)
t_float = torch.rand(B)
v_pred = fm_model(x0, t_float)
assert v_pred.shape == x0.shape, f"VelocityNet output shape: {v_pred.shape}"
# Gradient flow
v_pred.sum().backward()
has_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in fm_model.parameters())
assert has_grad, "No gradients flowing through VelocityNet"

# === Flow matching loss ===
torch.manual_seed(42)
fm_model2 = FlowMatchingVelocityNet(channels=C, time_dim=32)
loss_val = flow_matching_loss(fm_model2, x0)
assert loss_val.ndim == 0, "Loss should be a scalar"
assert loss_val.item() > 0, "Loss should be positive"
assert loss_val.requires_grad, "Loss should require grad"

# === Training reduces loss ===
torch.manual_seed(42)
fm_model3 = FlowMatchingVelocityNet(channels=C, time_dim=32)
fm_opt = torch.optim.Adam(fm_model3.parameters(), lr=1e-3)
x_data = torch.randn(8, C, H, W)
fm_losses = []
for _ in range(10):
    fm_opt.zero_grad()
    l = flow_matching_loss(fm_model3, x_data)
    l.backward()
    fm_opt.step()
    fm_losses.append(l.item())
assert fm_losses[-1] < fm_losses[0], f"Flow matching loss should decrease: {fm_losses[0]:.4f} → {fm_losses[-1]:.4f}"

# === ODE sampling ===
torch.manual_seed(42)
fm_model3.eval()
with torch.no_grad():
    fm_samples = ode_sample(fm_model3, (2, C, H, W), num_steps=20)
assert fm_samples.shape == (2, C, H, W), f"ODE sample shape wrong: {fm_samples.shape}"
assert not torch.isnan(fm_samples).any(), "ODE samples contain NaNs"
assert not torch.isinf(fm_samples).any(), "ODE samples contain Infs"

print("Problem 3: ALL TESTS PASSED")

---

## Problem 4: Sliding Window + Full Attention Hybrid

### Scenario

Long sequences need efficient attention. Build a hybrid attention system that alternates between sliding window attention (local, O(T·w)) and full attention (global, O(T²)). The sliding window handles local context cheaply, while periodic full attention layers capture long-range dependencies.

### Requirements

**`make_sliding_window_mask(seq_len, window_size) → mask`**
- Returns a boolean mask of shape `(seq_len, seq_len)` where `mask[i, j] = True` means position `i` CAN attend to position `j`
- Position `i` attends to positions in `[max(0, i - window_size), min(seq_len, i + window_size + 1)]`
- The mask is a band matrix along the diagonal

**`SlidingWindowAttention(nn.Module)`**
- Constructor: `__init__(self, embed_dim, num_heads, window_size)`
- Q, K, V projections via `nn.Linear(embed_dim, embed_dim)` each
- Output projection: `nn.Linear(embed_dim, embed_dim)`
- Forward: `(B, T, D)` → split into heads → compute `attn = Q @ K^T / sqrt(d_k)` → apply sliding window mask (set masked positions to `-inf` before softmax) → `attn @ V` → concat heads → output projection → `(B, T, D)`
- `d_k = embed_dim // num_heads`

**`FullAttention(nn.Module)`**
- Same interface as SlidingWindowAttention but without masking (standard multi-head self-attention)
- Constructor: `__init__(self, embed_dim, num_heads)`

**`HybridAttentionBlock(nn.Module)`**
- Constructor: `__init__(self, embed_dim, num_heads, num_layers, window_size, full_every_n=3)`
- Builds `num_layers` attention layers. Every `full_every_n`-th layer (0-indexed: layers 0, full_every_n, 2*full_every_n, ...) uses FullAttention. All others use SlidingWindowAttention.
- Each layer: LayerNorm → Attention → residual → LayerNorm → MLP (Linear→GELU→Linear) → residual
- Forward: apply all layers sequentially, return `(B, T, D)`

In [ ]:
def make_sliding_window_mask(seq_len: int, window_size: int) -> torch.Tensor:
    """Create boolean band mask for sliding window attention."""
    # YOUR CODE HERE
    pass


class SlidingWindowAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, window_size: int):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass


class FullAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass


class HybridAttentionBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, num_layers: int, window_size: int, full_every_n: int = 3):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests for Problem 4 ---
torch.manual_seed(42)

# === Sliding window mask ===
seq_len, w = 10, 2
mask = make_sliding_window_mask(seq_len, w)
assert mask.shape == (seq_len, seq_len), f"Mask shape: {mask.shape}"
assert mask.dtype == torch.bool, f"Mask dtype: {mask.dtype}"
# Diagonal should always be True
assert mask.diagonal().all(), "Diagonal should be True (attend to self)"
# Position 0 should attend to [0, 1, 2] (window_size=2)
assert mask[0, 0] and mask[0, 1] and mask[0, 2], "Position 0 should attend to 0, 1, 2"
assert not mask[0, 3], "Position 0 should NOT attend to position 3 with window=2"
# Position 5 should attend to [3, 4, 5, 6, 7]
for j in range(3, 8):
    assert mask[5, j], f"Position 5 should attend to {j}"
assert not mask[5, 2], "Position 5 should NOT attend to 2"
assert not mask[5, 8], "Position 5 should NOT attend to 8"

# Mask sparsity: sliding window has O(T*w) True entries vs O(T^2) for full
num_true = mask.sum().item()
full_entries = seq_len * seq_len
# For w=2, each row has at most 2*w+1=5 entries. So total ≈ T*(2w+1) = 50
assert num_true < full_entries, "Sliding window should be sparser than full attention"
expected_max = seq_len * (2 * w + 1)
assert num_true <= expected_max, f"Too many True entries: {num_true} > {expected_max}"

# === SlidingWindowAttention ===
torch.manual_seed(42)
embed_dim, num_heads = 64, 4
sw_attn = SlidingWindowAttention(embed_dim, num_heads, window_size=2)
x_seq = torch.randn(2, seq_len, embed_dim)
out_sw = sw_attn(x_seq)
assert out_sw.shape == x_seq.shape, f"SlidingWindowAttention output shape: {out_sw.shape}"

# Sliding window: changing a distant token should NOT affect output at position 0
torch.manual_seed(42)
x_modified = x_seq.clone()
x_modified[0, 8, :] = torch.randn(embed_dim)  # position 8 is far from position 0 with w=2
with torch.no_grad():
    out_orig = sw_attn(x_seq)
    out_mod = sw_attn(x_modified)
assert torch.allclose(out_orig[0, 0], out_mod[0, 0], atol=1e-5), \
    "Sliding window: distant token change should NOT affect position 0"
# But a nearby token change SHOULD affect it
x_near = x_seq.clone()
x_near[0, 1, :] = torch.randn(embed_dim)  # position 1 is within window of position 0
with torch.no_grad():
    out_near = sw_attn(x_near)
assert not torch.allclose(out_orig[0, 0], out_near[0, 0], atol=1e-5), \
    "Sliding window: nearby token change SHOULD affect position 0"

# === FullAttention ===
full_attn = FullAttention(embed_dim, num_heads)
out_full = full_attn(x_seq)
assert out_full.shape == x_seq.shape, f"FullAttention output shape: {out_full.shape}"
# Full attention: changing ANY token should affect output at position 0
with torch.no_grad():
    out_f1 = full_attn(x_seq)
    out_f2 = full_attn(x_modified)  # changed position 8
assert not torch.allclose(out_f1[0, 0], out_f2[0, 0], atol=1e-5), \
    "Full attention: any token change SHOULD affect position 0"

# === HybridAttentionBlock ===
torch.manual_seed(42)
hybrid = HybridAttentionBlock(embed_dim, num_heads, num_layers=6, window_size=2, full_every_n=3)
out_hybrid = hybrid(x_seq)
assert out_hybrid.shape == x_seq.shape, f"Hybrid output shape: {out_hybrid.shape}"

# Gradient flow
out_hybrid.sum().backward()
has_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in hybrid.parameters())
assert has_grad, "No gradients flowing through HybridAttentionBlock"

print("Problem 4: ALL TESTS PASSED")

---

## Problem 5: Progressive Distillation

### Scenario

DDPM sampling is slow (1000 steps). Progressive distillation trains a student model to match the teacher's 2-step output in 1 step, halving the required steps each round. Build the complete distillation system.

### Requirements

**`teacher_two_step(teacher, x_t, t, schedule) → x_result`**
- Perform 2 DDPM denoising steps starting from `x_t` at timestep `t`
- Step 1: denoise from `t` to `t-1` using DDPM update rule:
  - `noise_pred = teacher(x_t, t_emb)`
  - `x_{t-1} = sqrt_recip_alpha[t] * (x_t - betas[t] / sqrt_one_minus_alpha_bar[t] * noise_pred)`
  - If `t-1 > 0`: add `sqrt(posterior_variance[t]) * noise`
- Step 2: denoise from `t-1` to `t-2` using same formula
  - If `t < 2`, skip second step (return after one step)
- Use `sinusoidal_embedding` from Problem 1 for time embeddings
- Return the result after 2 steps (or 1 if `t < 2`)

**`StudentDenoiser(nn.Module)`**
- Same architecture as `SimpleUNet` from Problem 1 (you can reuse it)
- The student learns to jump 2 steps at once

**`distillation_step(teacher, student, x0, schedule, optimizer, T) → loss`**
- Sample random `t ~ Uniform{2, ..., T-1}` for each batch item (need at least 2 steps of room)
- Sample noise and compute `x_t = q_sample(x0, t, schedule)`
- **Teacher target (no grad):** run `teacher_two_step(teacher, x_t, t, schedule)` to get target output
- **Student prediction:** student does 1 denoising step from `t` to `t-2`:
  - `noise_pred = student(x_t, t_emb)`
  - Apply DDPM formula but stepping by 2: use schedule values at index `t` but reconstruct toward `t-2`
  - Simpler: just have student predict noise, then compare resulting denoised outputs
- Loss: `MSE(student_output, teacher_target)` where both are the denoised results
- Backprop through student only
- Return loss as float

**Simplified approach (recommended):**
- Teacher target: `with torch.no_grad(): target = teacher_two_step(teacher, x_t.detach(), t, schedule)`
- Student prediction: `pred = student_one_step(student, x_t, t, schedule)` — one DDPM step
- The student is trained so that its single step output matches teacher's two-step output
- `student_one_step` applies the same DDPM formula as one step of `teacher_two_step`
- Loss = MSE(pred, target)

In [ ]:
def teacher_two_step(teacher: nn.Module, x_t: torch.Tensor, t: torch.Tensor,
                     schedule: dict) -> torch.Tensor:
    """Run 2 DDPM denoising steps with the teacher model."""
    # YOUR CODE HERE
    pass


def student_one_step(student: nn.Module, x_t: torch.Tensor, t: torch.Tensor,
                     schedule: dict) -> torch.Tensor:
    """Run 1 DDPM denoising step with the student model."""
    # YOUR CODE HERE
    pass


def distillation_step(teacher: nn.Module, student: nn.Module, x0: torch.Tensor,
                      schedule: dict, optimizer: torch.optim.Optimizer, T: int) -> float:
    """One distillation training step: student matches teacher's 2-step output in 1 step."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 5 ---
# NOTE: This problem reuses sinusoidal_embedding, make_schedule, q_sample, SimpleUNet from Problem 1.
# Solve Problem 1 first.

torch.manual_seed(42)
T_distill = 50
sched_d = make_schedule(T_distill)

teacher = SimpleUNet(time_dim=32)
student = SimpleUNet(time_dim=32)
# Initialize student differently from teacher
for p in student.parameters():
    nn.init.normal_(p, 0, 0.02)

B_d = 4
x0_d = torch.randn(B_d, 1, 16, 16)

# === teacher_two_step output shape ===
torch.manual_seed(42)
t_test_d = torch.full((B_d,), 10, dtype=torch.long)
x_t_test = q_sample(x0_d, t_test_d, sched_d)
teacher.eval()
with torch.no_grad():
    teacher_out = teacher_two_step(teacher, x_t_test, t_test_d, sched_d)
assert teacher_out.shape == x0_d.shape, f"Teacher output shape: {teacher_out.shape}"
assert not torch.isnan(teacher_out).any(), "Teacher output has NaNs"
assert not torch.isinf(teacher_out).any(), "Teacher output has Infs"

# === student_one_step output shape ===
student.train()
student_out = student_one_step(student, x_t_test, t_test_d, sched_d)
assert student_out.shape == x0_d.shape, f"Student output shape: {student_out.shape}"
assert not torch.isnan(student_out).any(), "Student output has NaNs"

# === teacher_two_step at t=1 should handle edge case (only 1 step possible) ===
t_low = torch.full((B_d,), 1, dtype=torch.long)
x_t_low = q_sample(x0_d, t_low, sched_d)
with torch.no_grad():
    teacher_low = teacher_two_step(teacher, x_t_low, t_low, sched_d)
assert teacher_low.shape == x0_d.shape, "teacher_two_step should handle t=1 edge case"
assert not torch.isnan(teacher_low).any(), "teacher_two_step at t=1 has NaNs"

# === teacher_two_step is deterministic without noise (eval mode) ===
torch.manual_seed(99)
with torch.no_grad():
    out_a = teacher_two_step(teacher, x_t_test, t_test_d, sched_d)
torch.manual_seed(99)
with torch.no_grad():
    out_b = teacher_two_step(teacher, x_t_test, t_test_d, sched_d)
assert torch.allclose(out_a, out_b, atol=1e-5), "teacher_two_step should be deterministic with same seed"

# === Distillation loss decreases ===
torch.manual_seed(42)
teacher2 = SimpleUNet(time_dim=32)
student2 = SimpleUNet(time_dim=32)
teacher2.eval()
student2.train()
opt_d = torch.optim.Adam(student2.parameters(), lr=1e-3)
x_train_d = torch.randn(8, 1, 16, 16)
d_losses = []
for _ in range(10):
    l = distillation_step(teacher2, student2, x_train_d, sched_d, opt_d, T_distill)
    d_losses.append(l)
assert isinstance(d_losses[0], float), "distillation_step should return float"
assert d_losses[-1] < d_losses[0], f"Distillation loss should decrease: {d_losses[0]:.4f} → {d_losses[-1]:.4f}"

# === Teacher params should not change during distillation ===
teacher_params_before = {n: p.clone() for n, p in teacher2.named_parameters()}
_ = distillation_step(teacher2, student2, x_train_d, sched_d, opt_d, T_distill)
for name, p in teacher2.named_parameters():
    assert torch.equal(p, teacher_params_before[name]), f"Teacher param {name} changed during distillation!"

# === Student output should approach teacher over training ===
torch.manual_seed(123)
t_eval = torch.full((B_d,), 20, dtype=torch.long)
x_t_eval = q_sample(x0_d, t_eval, sched_d)
with torch.no_grad():
    teacher_ref = teacher_two_step(teacher2, x_t_eval, t_eval, sched_d)
    student_ref = student_one_step(student2, x_t_eval, t_eval, sched_d)
dist = F.mse_loss(student_ref, teacher_ref).item()
assert dist < 100, f"After training, student-teacher distance should be reasonable: {dist}"

print("Problem 5: ALL TESTS PASSED")

---

## Problem 6: Complete Eval Suite

### Scenario

Build a complete evaluation suite for generative models. Compute FID (distribution distance), CLIP score (text-image alignment), LPIPS approximation (perceptual distance), and package them into a structured report.

### Requirements

**`compute_fid_stats(features) → (mu, sigma)`**
- Input: `features` of shape `(N, D)` — feature vectors from a backbone
- `mu`: mean across samples, shape `(D,)`
- `sigma`: covariance matrix, shape `(D, D)`. Use unbiased estimator.
- Hint: center features, then `sigma = X^T @ X / (N - 1)` where X is centered

**`matrix_sqrt(M) → M^{1/2}`**
- Compute matrix square root via eigendecomposition
- `eigenvalues, eigenvectors = torch.linalg.eigh(M)` (M must be symmetric)
- Clamp eigenvalues to >= 0 (numerical stability)
- `M^{1/2} = V @ diag(sqrt(eigenvalues)) @ V^T`
- Return real-valued result

**`frechet_distance(mu1, sigma1, mu2, sigma2) → float`**
- Formula: `FD = ||mu1 - mu2||^2 + Tr(sigma1 + sigma2 - 2 * sqrtm(sigma1 @ sigma2))`
- Use `matrix_sqrt` for the square root term
- Product `sigma1 @ sigma2` may not be symmetric. Compute `sqrtm(sigma1 @ sigma2)` as:
  - `covmean = matrix_sqrt(sigma1 @ sigma2)` — if this fails due to non-symmetry, symmetrize: `M = (sigma1 @ sigma2 + (sigma1 @ sigma2).T) / 2`
- Return scalar float

**`clip_score(image_features, text_features) → float`**
- Both inputs: `(N, D)` normalized feature vectors
- Compute cosine similarity between corresponding pairs: `sim[i] = dot(img[i], txt[i])`
- L2-normalize both inputs first (in case they aren't already)
- Return mean similarity as float

**`lpips_approx(features_a, features_b) → float`**
- Approximate LPIPS as mean L2 distance between L2-normalized feature vectors
- Normalize each feature vector to unit norm
- Compute per-pair L2 distance: `||norm_a[i] - norm_b[i]||_2`
- Return mean distance as float

**`EvalReport`**
- Constructor: `__init__(self, fid_threshold=50.0, clip_threshold=0.25, lpips_threshold=0.5)`
- Method: `compute(real_features, gen_features, image_features, text_features, features_a, features_b) → dict`
  - `real_features`, `gen_features`: `(N, D)` for FID
  - `image_features`, `text_features`: `(N, D)` for CLIP score
  - `features_a`, `features_b`: `(N, D)` for LPIPS
  - Returns dict:
    ```python
    {
        "fid": {"value": float, "good": bool},  # good if value < fid_threshold
        "clip_score": {"value": float, "good": bool},  # good if value > clip_threshold
        "lpips": {"value": float, "good": bool},  # good if value < lpips_threshold
        "overall": bool  # True if ALL metrics are "good"
    }
    ```

In [ ]:
def compute_fid_stats(features: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """Compute mean and covariance of feature vectors."""
    # YOUR CODE HERE
    pass


def matrix_sqrt(M: torch.Tensor) -> torch.Tensor:
    """Matrix square root via eigendecomposition."""
    # YOUR CODE HERE
    pass


def frechet_distance(mu1: torch.Tensor, sigma1: torch.Tensor,
                     mu2: torch.Tensor, sigma2: torch.Tensor) -> float:
    """Frechet Inception Distance between two Gaussians."""
    # YOUR CODE HERE
    pass


def clip_score(image_features: torch.Tensor, text_features: torch.Tensor) -> float:
    """Mean cosine similarity between image-text pairs."""
    # YOUR CODE HERE
    pass


def lpips_approx(features_a: torch.Tensor, features_b: torch.Tensor) -> float:
    """Approximate LPIPS as mean L2 distance between normalized features."""
    # YOUR CODE HERE
    pass


class EvalReport:
    def __init__(self, fid_threshold: float = 50.0, clip_threshold: float = 0.25, lpips_threshold: float = 0.5):
        # YOUR CODE HERE
        pass

    def compute(self, real_features: torch.Tensor, gen_features: torch.Tensor,
                image_features: torch.Tensor, text_features: torch.Tensor,
                features_a: torch.Tensor, features_b: torch.Tensor) -> dict:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests for Problem 6 ---
torch.manual_seed(42)

N, D = 100, 64

# === FID stats ===
features = torch.randn(N, D)
mu, sigma = compute_fid_stats(features)
assert mu.shape == (D,), f"Mean shape: {mu.shape}"
assert sigma.shape == (D, D), f"Covariance shape: {sigma.shape}"
# Covariance should be symmetric
assert torch.allclose(sigma, sigma.T, atol=1e-5), "Covariance should be symmetric"
# Mean should be close to 0 for standard normal features
assert mu.abs().max() < 0.5, f"Mean of standard normal features too far from 0: {mu.abs().max()}"

# === Matrix sqrt ===
A = torch.randn(D, D)
M_spd = A @ A.T + 0.1 * torch.eye(D)  # symmetric positive definite
M_root = matrix_sqrt(M_spd)
assert M_root.shape == (D, D), f"Matrix sqrt shape: {M_root.shape}"
# Verify: M_root @ M_root ≈ M_spd
reconstructed = M_root @ M_root
assert torch.allclose(reconstructed, M_spd, atol=1e-3), "matrix_sqrt(M) @ matrix_sqrt(M) should ≈ M"

# === FID of identical distributions ≈ 0 ===
torch.manual_seed(42)
real_feats = torch.randn(500, D)
# Same distribution, different samples
gen_feats_same = torch.randn(500, D)
mu_r, sig_r = compute_fid_stats(real_feats)
mu_g, sig_g = compute_fid_stats(gen_feats_same)
fid_same = frechet_distance(mu_r, sig_r, mu_g, sig_g)
assert isinstance(fid_same, float), f"FID should be float: {type(fid_same)}"
# FID between same-distribution samples should be small (not exactly 0 due to finite samples)
assert fid_same < 10.0, f"FID between same-distribution samples too high: {fid_same}"

# FID with shifted distribution should be larger
gen_feats_shifted = torch.randn(500, D) + 3.0
mu_gs, sig_gs = compute_fid_stats(gen_feats_shifted)
fid_shifted = frechet_distance(mu_r, sig_r, mu_gs, sig_gs)
assert fid_shifted > fid_same, f"FID of shifted dist should be larger: {fid_shifted} vs {fid_same}"

# FID of exactly identical stats = 0
fid_exact = frechet_distance(mu_r, sig_r, mu_r, sig_r)
assert abs(fid_exact) < 1e-3, f"FID of identical stats should be ≈ 0: {fid_exact}"

# === CLIP score ===
# Matched pairs (same direction) should have high similarity
img_feats = F.normalize(torch.randn(N, D), dim=-1)
txt_feats_matched = img_feats + 0.1 * torch.randn(N, D)  # slightly noisy copies
clip_matched = clip_score(img_feats, txt_feats_matched)
assert isinstance(clip_matched, float), f"CLIP score should be float: {type(clip_matched)}"

# Random pairs should have lower similarity
txt_feats_random = F.normalize(torch.randn(N, D), dim=-1)
clip_random = clip_score(img_feats, txt_feats_random)
assert clip_matched > clip_random, f"Matched CLIP > random CLIP: {clip_matched} vs {clip_random}"

# Perfect match
clip_perfect = clip_score(img_feats, img_feats)
assert abs(clip_perfect - 1.0) < 1e-5, f"CLIP of identical features should be 1.0: {clip_perfect}"

# === LPIPS ===
# Identical features should give 0
lpips_same = lpips_approx(img_feats, img_feats)
assert isinstance(lpips_same, float), f"LPIPS should be float: {type(lpips_same)}"
assert abs(lpips_same) < 1e-5, f"LPIPS of identical features should be ≈ 0: {lpips_same}"

# Different features should give > 0
other_feats = F.normalize(torch.randn(N, D), dim=-1)
lpips_diff = lpips_approx(img_feats, other_feats)
assert lpips_diff > 0, f"LPIPS of different features should be > 0: {lpips_diff}"

# === EvalReport ===
report = EvalReport(fid_threshold=50.0, clip_threshold=0.25, lpips_threshold=0.5)
result = report.compute(
    real_features=real_feats,
    gen_features=gen_feats_same,
    image_features=img_feats,
    text_features=txt_feats_matched,
    features_a=img_feats,
    features_b=img_feats,
)
assert isinstance(result, dict), "EvalReport.compute should return dict"
assert "fid" in result and "clip_score" in result and "lpips" in result and "overall" in result
assert isinstance(result["fid"]["value"], float), "FID value should be float"
assert isinstance(result["fid"]["good"], bool), "FID 'good' should be bool"
assert isinstance(result["overall"], bool), "overall should be bool"
# With good inputs, all should pass
assert result["fid"]["good"], f"FID should be good: {result['fid']['value']}"
assert result["lpips"]["good"], f"LPIPS should be good (identical features): {result['lpips']['value']}"
assert result["overall"], f"Overall should be True with good metrics: {result}"

print("Problem 6: ALL TESTS PASSED")

---

## Scoring

| Result | Assessment |
|--------|------------|
| Solved in < 25 min, all tests pass | **Overqualified** — you're ready for anything |
| Solved in 25-30 min, minor debugging | **Strong pass** — these are at the ceiling of CoderPad difficulty |
| 30-40 min or needed hints | **Pass** — you can handle hard problems, keep practicing |
| Couldn't solve or > 40 min | **Go back to HARD** — nail those first, then return here |